# NUFROST Sparse Observation Sweep (Colab)

This notebook systematically evaluates the performance of NUFROST, Zhu2015, and HANTS algorithms under different levels of random point masking (sparse observations).

**Number of masked points to scan:** 1000, 5000, 10000, 20000, 50000, 100000 points.

For each image chunk (band, lon, lat), the notebook runs the random‑point evaluation (`evaluate_algorithms`) with each `num_points`, collects the metrics, and appends them incrementally to a CSV file.
Resume‑from‑previous is supported by checking the output CSV.

**Note:** The actual masking ratio depends on the total number of valid spatio‑temporal points in the cube. The notebook records both `NumPoints` and, if feasible, the computed masking ratio. For consistent trends across images, `NumPoints` serves as a proxy for sparsity.

## 1. Configuration

In [ ]:
from pathlib import Path

MOUNT_POINT_IN_COLAB = Path("/content/drive")
PROJECT_PATH_IN_GDRIVE = Path("WorkSpaces/nufrost")
PROJECT_DIR = MOUNT_POINT_IN_COLAB / "MyDrive" / PROJECT_PATH_IN_GDRIVE
IMAGE_DIR   = PROJECT_DIR / "data/hls"          # or "data/sentinel-2"
OUTPUT_DIR  = PROJECT_DIR / "data/output"
CACHE_DIR   = PROJECT_DIR / "data/cache/colab"
OUTPUT_CSV_PATH = OUTPUT_DIR / "sparse_observation_sweep_results.csv"

# If you want to limit to specific files, list them here; leave empty to auto‑discover.
IMAGE_NAMES = []

# Number of random points to mask (these are absolute counts, not ratios)
NUM_POINTS_LIST = [1000, 5000, 10000, 20000, 50000, 100000]

# Parallel jobs (-1 uses all available cores)
N_JOBS = -1

# Random seed for reproducibility (base seed, will be modified per image/num_points)
BASE_SEED = 42

## 2. Mount Google Drive

In [ ]:
import os
from google.colab import drive # type: ignore[import]

drive.mount(MOUNT_POINT_IN_COLAB.as_posix())
os.chdir(PROJECT_DIR)
print(f"[Working directory changed to: {os.getcwd()}]")


## 3. Install Dependencies

In [ ]:
!apt-get install -y gdal-bin
%pip install -r requirements.txt


## 4. Import Modules

In [ ]:
import src.data_loader
import importlib
import src.evaluation
import pandas as pd
import glob
import re
import numpy as np
import time

from config import build_args
from IPython.display import display

importlib.reload(src.nufrost)
importlib.reload(src.zhu2015)
importlib.reload(src.hants)
importlib.reload(src.evaluation)
importlib.reload(src.data_loader)


## 5. Auto‑discover Image Chunks

In [ ]:
if IMAGE_NAMES:
    image_paths_list = [[(IMAGE_DIR / name).as_posix()] for name in IMAGE_NAMES]
else:
    # Auto‑detect all distinct coordinates and bands, then construct their VRTs
    files = glob.glob((IMAGE_DIR / "*.tif").as_posix())
    loc_ids = set()
    for f in files:
        f_name = Path(f).name
        match = re.search(r"_([A-Z0-9]+)_lon([0-9.]+)_lat([0-9.]+).*?(?:_part\d+)?(?:-\d{10}-\d{10})?\.tif$", f_name)
        if match:
            band = match.group(1)
            lon = float(match.group(2))
            lat = float(match.group(3))
            loc_ids.add((band, lon, lat))

    image_paths_list = []
    for band, lon, lat in loc_ids:
        # find_image_chunks returns the ordered VRT paths for one coordinate/band
        chunks = src.data_loader.find_image_chunks(IMAGE_DIR.as_posix(), lon, lat, band, cache_dir=CACHE_DIR.as_posix())
        if chunks:
            image_paths_list.append(chunks)

print(f"Found {len(image_paths_list)} distinct spatial/band chunks to evaluate.")
if not image_paths_list:
    print("No image chunks found. Please check IMAGE_DIR and filename patterns.")


## 6. Run Sparse Observation Sweep

In [ ]:
from pathlib import Path

# Ensure output directory exists
Path(OUTPUT_CSV_PATH).parent.mkdir(parents=True, exist_ok=True)

# Load previously evaluated results (if any) to resume
evaluated = set()
if OUTPUT_CSV_PATH.exists():
    try:
        existing_df = pd.read_csv(OUTPUT_CSV_PATH)
        for _, row in existing_df.iterrows():
            evaluated.add((row["Image"], row["NumPoints"]))
        print(f"Found existing results for {len(existing_df)} rows. Resuming...")
    except Exception as e:
        print(f"Could not read existing CSV: {e}")

all_results = []

for image_paths in image_paths_list:
    first_path = Path(image_paths[0])
    match = re.search(r"([A-Z0-9]+_lon[0-9.]+_lat[0-9.]+)", first_path.stem)
    loc_id = match.group(1) if match else first_path.stem

    print(f"\n--- Evaluating: {loc_id} ---")

    for num_points in NUM_POINTS_LIST:
        if (loc_id, num_points) in evaluated:
            print(f"    Skipping {num_points} points (already evaluated)")
            continue

        print(f"    Running {num_points} points")

        # Build args with default config
        args = build_args({})
        args.image = image_paths
        args.cache_dir = CACHE_DIR.as_posix()
        args.n_jobs = N_JOBS
        args.force_refresh = False

        # Set random seed for reproducibility (different per image/num_points)
        seed = BASE_SEED + hash(loc_id) % 1000 + num_points
        np.random.seed(seed)

        start_time = time.time()
        df_results = src.evaluation.evaluate_algorithms(
            image_path=args.image,
            args=args,
            num_points=num_points,
            n_jobs=args.n_jobs,
        )

        elapsed = time.time() - start_time
        print(f"      finished in {elapsed:.1f}s")

        if df_results.empty:
            print(f"      WARNING: No results for {num_points} points")
            continue

        # Add metadata columns
        df_results["Image"] = loc_id
        df_results["NumPoints"] = num_points
        df_results["Seed"] = seed

        all_results.append(df_results)

        # Incremental save after each num_points
        Path(OUTPUT_CSV_PATH).parent.mkdir(parents=True, exist_ok=True)
        header = not OUTPUT_CSV_PATH.exists()
        df_results.to_csv(OUTPUT_CSV_PATH, mode="a", header=header, index=False)

print("\n========== Sparse Observation Sweep Complete ==========")
print(f"Results saved to: {OUTPUT_CSV_PATH}")


## 7. Quick Summary (Optional)

In [ ]:
if OUTPUT_CSV_PATH.exists():
    df_all = pd.read_csv(OUTPUT_CSV_PATH)
    print(f"Total rows collected: {len(df_all)}")
    print("\nMean metrics per algorithm and number of masked points:")
    summary = df_all.groupby(["Algorithm", "NumPoints"])[["RMSE", "MAE", "R", "OutlierRatio"]].mean().round(4)
    display(summary)
else:
    print("No results CSV found.")
